In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('imdb_top_1000.csv')

In [ ]:
df.head(10)

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"
5,https://m.media-amazon.com/images/M/MV5BNzA5ZD...,The Lord of the Rings: The Return of the King,2003,U,201 min,"Action, Adventure, Drama",8.9,Gandalf and Aragorn lead the World of Men agai...,94.0,Peter Jackson,Elijah Wood,Viggo Mortensen,Ian McKellen,Orlando Bloom,1642758,"377,845,905"
6,https://m.media-amazon.com/images/M/MV5BNGNhMD...,Pulp Fiction,1994,A,154 min,"Crime, Drama",8.9,"The lives of two mob hitmen, a boxer, a gangst...",94.0,Quentin Tarantino,John Travolta,Uma Thurman,Samuel L. Jackson,Bruce Willis,1826188,"107,928,762"
7,https://m.media-amazon.com/images/M/MV5BNDE4OT...,Schindler's List,1993,A,195 min,"Biography, Drama, History",8.9,"In German-occupied Poland during World War II,...",94.0,Steven Spielberg,Liam Neeson,Ralph Fiennes,Ben Kingsley,Caroline Goodall,1213505,"96,898,818"
8,https://m.media-amazon.com/images/M/MV5BMjAxMz...,Inception,2010,UA,148 min,"Action, Adventure, Sci-Fi",8.8,A thief who steals corporate secrets through t...,74.0,Christopher Nolan,Leonardo DiCaprio,Joseph Gordon-Levitt,Elliot Page,Ken Watanabe,2067042,"292,576,195"
9,https://m.media-amazon.com/images/M/MV5BMmEzNT...,Fight Club,1999,A,139 min,Drama,8.8,An insomniac office worker and a devil-may-car...,66.0,David Fincher,Brad Pitt,Edward Norton,Meat Loaf,Zach Grenier,1854740,"37,030,102"


In [ ]:
df.columns

Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='object')

In [ ]:

# compute similarity between two movies
def sim(title_1:str, title_2:str,  # titles
        movies_df, # movies data frame
        genre_w:float=1, # genre
        dir_w:float=1,
        ryear_w:float=1,
        rating_w:float=1,
        stars_w:float=1 # starts
        )->float:

    # get the rows for the 2 movies from the df
    info1=movies_df.loc[movies_df.Series_Title==title_1]
    info2=movies_df.loc[movies_df.Series_Title==title_2]

    # get the release years
    ryear1=int(info1['Released_Year'].values[0])
    ryear2=int(info2['Released_Year'].values[0])

    # get the genres
    genres1=set([x.strip() for x in info1['Genre'].values[0].split(',')])
    genres2=set([x.strip() for x in info2['Genre'].values[0].split(',')])

    # get the directors
    dir1=set([x.strip() for x in info1['Director'].values[0].split(',')])
    dir2=set([x.strip() for x in info2['Director'].values[0].split(',')])

    # get the avg imdb ratings
    rating1=float(info1['IMDB_Rating'].values[0])
    rating2=float(info2['IMDB_Rating'].values[0])

    # get the list of actors
    star_list1=set([info1['Star'+str(i)].values[0].strip() for i in range(1,5)])
    star_list2=set([info2['Star'+str(i)].values[0].strip() for i in range(1,5)])

    # compute the jaccard for the actors
    star_jacc=len(star_list1.intersection(star_list2))/len(star_list1.union(star_list2))

    # jaccard for directors, genres
    dir_jacc=len(dir1.intersection(dir2))/len(dir1.union(dir2))
    genre_jacc=len(genres1.intersection(genres2))/len(genres1.union(genres2))

    # difference in release years
    ryear_diff=abs(ryear1-ryear2)/100

    # normalized imdb rating
    rating_norm=rating1/10

    # return final similarity
    return genre_w*genre_jacc+\
              dir_w*dir_jacc+\
              ryear_w*(1-ryear_diff)+\
              rating_w*rating_norm+\
              stars_w*star_jacc


In [ ]:
sim('Batman Begins', 'Iron Man',df, stars_w=1)

2.4566666666666666

In [ ]:
def recommend(input_title:str,
              k:int,
              movies_df,
              genre_w:float=1,
              dir_w:float=1,
              ryear_w:float=1,
              rating_w:float=1,
              stars_w:float=1
              )->list:

  results={}

  for title in movies_df.Series_Title:# for each movie

    try:
      my_sim=sim(title,input_title,movies_df,genre_w,dir_w,ryear_w,rating_w,stars_w) # compute the simliarity with the input movie
    except Exception as ex:
      print(ex)
      print('sim failed, skipping',title )

    results[title]=my_sim

  return sorted(results.items(),key=lambda x:x[1],reverse=True)[:k] # sort and return the top k

In [ ]:
recommend('Toy Story',10,df,rating_w=0)

invalid literal for int() with base 10: 'PG'
sim failed, skipping Apollo 13


[('Toy Story', 4.0),
 ('Toy Story 2', 3.2933333333333334),
 ('Toy Story 3', 2.1833333333333336),
 ('Toy Story 4', 2.0933333333333333),
 ('Aladdin', 1.97),
 ('Kurenai no buta', 1.97),
 ('Monsters, Inc.', 1.94),
 ('Shrek', 1.94),
 ('Who Framed Roger Rabbit', 1.93),
 ('Finding Nemo', 1.92)]